# Orquestador Silver y DLQ

Ejecuta la creación de schemas y las tres transformaciones en orden. Bronze se lee solamente; la ingesta no pertenece a este flujo.

In [ ]:
import json
import time
import uuid

dbutils.widgets.text("catalog", "steam_game_intelligence")
dbutils.widgets.text("bronze_schema", "bronze")
dbutils.widgets.text("silver_schema", "silver")
dbutils.widgets.text("dlq_schema", "dlq")
dbutils.widgets.text("audit_schema", "audit")
dbutils.widgets.text("run_ingestion", "false")
dbutils.widgets.text("cutoff_us", "")
dbutils.widgets.text("run_id", "")

catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
silver_schema = dbutils.widgets.get("silver_schema")
dlq_schema = dbutils.widgets.get("dlq_schema")
audit_schema = dbutils.widgets.get("audit_schema")
run_ingestion = dbutils.widgets.get("run_ingestion").lower()
run_id = dbutils.widgets.get("run_id") or str(uuid.uuid4())

if run_ingestion != "false":
    raise ValueError("run_ingestion debe ser false: Bronze es de solo lectura")
cutoff_us = dbutils.widgets.get("cutoff_us") or str(int(time.time() * 1_000_000))
if not cutoff_us.isdigit():
    raise ValueError("cutoff_us debe ser un epoch en microsegundos")

base_args = {
    "catalog": catalog,
    "bronze_schema": bronze_schema,
    "silver_schema": silver_schema,
    "dlq_schema": dlq_schema,
    "audit_schema": audit_schema,
    "cutoff_us": cutoff_us,
    "run_id": run_id,
}
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
root = notebook_path.rsplit("/src/", 1)[0]
transform_path = f"{root}/src/transformaciones"
dbutils.notebook.run(f"{transform_path}/silver_crear_schemas", 0, base_args)
for name in ["silver_games", "silver_player_counts_monthly", "silver_twitch_activity_monthly"]:
    result = json.loads(dbutils.notebook.run(f"{transform_path}/{name}", 0, base_args))
    if result.get("status") != "completed":
        raise RuntimeError(f"La transformación {name} no terminó correctamente: {result}")

dbutils.notebook.exit(json.dumps({"status": "completed", "run_id": run_id, "cutoff_us": int(cutoff_us)}))